In [2]:
import pandas as pd
import numpy as np

Input = "SPOC_analysis_results_poco.csv"
Output = "output_with_flags.csv"

POWER_THRESHOLD = 0.05
PERIOD_TOL_FRAC = 0.1
DOUBLE_DIP_RATIO = 0.5

# Column names (change if needed)
P1 = "best_period_days"
POW1 = "max_power"
P2 = "peak2_period"
POW2 = "peak2_power"

df = pd.read_csv(Input)

#Power threshold
df['power_pass'] = df[POW1] > POWER_THRESHOLD


#Period agreement per TIC
def compute_period_agreement(group):
    idx_ref = group[POW1].idxmax()
    ref_period = group.loc[idx_ref, P1]

    frac_diff = np.abs(group[P1] - ref_period) / ref_period
    group['period_agree'] = frac_diff <= PERIOD_TOL_FRAC
    group['ref_period'] = ref_period  
    return group

df = df.groupby('ticid', group_keys=False).apply(compute_period_agreement)


#Combined column
df['power_and_agree'] = df['power_pass'] & df['period_agree']


#Double-dipper flag
# Guard against missing values
df['double_dipper'] = (
    df[POW2].fillna(0) >= DOUBLE_DIP_RATIO * df[POW1]
)

# Optional: also require peaks to be real detections
# df['double_dipper'] &= df[POW2] > 0.01


#Save
df.to_csv(Output, index=False)
print("Saved:", Output)

Saved: output_with_flags.csv
